Code for defining the weighted area

In [ ]:
import xarray as xr
import numpy as np

# Open one of your CICE history files (adjust path/filename as needed)
ds = xr.open_dataset("/work/cmcc/fc09621/EXPS/CGLORS/archive/test0/ice/hist/test0.cice.h.2010-01-02.nc")
ds_grid = xr.open_dataset("/data/cmcc/is14120/osisaf/orca_202001_nh.nc")


# Extract the grid cell area
area = ds["tarea"]  # shape (nj, ni)
mask = ds["tmask"] 

# Compute current total, only ocean grid
area_ocean = area.where(mask == 1)
current_sum = area_ocean.sum().item()


# Target sum
target_sum = 1/120

# Compute scaling factor
scale = target_sum / current_sum

# Apply scaling
scaled_area = area_ocean * scale


# Create a new Dataset for saving
ds_out = xr.Dataset(
    {
        "tarea_scaled": (("nj", "ni"), scaled_area.data)
    },
    coords={
        "TLON": (("nj", "ni"), ds["TLON"].data),
        "TLAT": (("nj", "ni"), ds["TLAT"].data),
    },
    attrs={"description": "Rescaled grid cell area so that sum = 1/120"}   # 120 is the number of timesteps, i.e. 12 months * 10 years
)

# Save to NetCDF
out_file = "/work/cmcc/fc09621/EXPS/CGLORS/scratch/scaled_tarea_ts.nc"
ds_out.to_netcdf(out_file)

print(f"Scaled tarea written to {out_file}")

Scaled tarea written to /work/cmcc/fc09621/EXPS/CGLORS/scratch/scaled_tarea_ts.nc


Script below to crop a subset with lat > 55

In [3]:
# Open one of your CICE history files (adjust path/filename as needed)
ds = xr.open_dataset("/work/cmcc/fc09621/EXPS/CGLORS/archive/test0/ice/hist/test0.cice.h.2010-01-02.nc")
ds = ds.rename({'nj': 'y', 'ni': 'x'})

ds_grid = xr.open_dataset("/data/cmcc/is14120/osisaf/orca_202001_nh.nc")


# define a mask for latitude greater than 55
masked_ds = (ds_grid.nav_lat > 55).compute()

# Extract the grid cell area
area = ds["tarea"].where(masked_ds, drop=True)  # shape (nj, ni)
mask = ds["tmask"].where(masked_ds, drop=True) 

lon = ds["TLON"].where(masked_ds, drop=True)  # shape (nj, ni)
lat = ds["TLAT"].where(masked_ds, drop=True) 

# Compute current total, only ocean grid
area_ocean = area.where(mask == 1)
current_sum = area_ocean.sum().item()

# Target sum
target_sum = 1/120

# Compute scaling factor
scale = target_sum / current_sum

# Apply scaling
scaled_area = area_ocean * scale


# Create a new Dataset for saving
ds_out = xr.Dataset(
    {
        "tarea_scaled": (("y", "x"), scaled_area.data)
    },
    coords={
        "TLON": (("y", "x"), lon.data),
        "TLAT": (("y", "x"), lat.data),
    },
    attrs={"description": "Cut domain lat > 55 and rescale grid cell area so that sum = 1/120"}
)

# Save to NetCDF
out_file = "/work/cmcc/fc09621/EXPS/CGLORS/scratch/scaled_tarea_ts_cut.nc"
ds_out.to_netcdf(out_file)

print(f"Scaled tarea written to {out_file}")

Scaled tarea written to /work/cmcc/fc09621/EXPS/CGLORS/scratch/scaled_tarea_ts_cut.nc
